# VinDr-CXR-VQA — MedGemma-4B-it fine-tuning on Kaggle

Reproduces arXiv:2511.00504v2 — *VinDr-CXR-VQA: A Visual Question Answering
Dataset for Explainable Chest X-ray Analysis with Multi-Task Learning*.

**Before running**

1. **Attach the data.** Upload `vindr-cxr-vqa-kaggle.zip`
   (`python scripts/make_kaggle_bundle.py`) as a Kaggle Dataset and attach it
   via *Add Input*. Paste its path into `BUNDLE` in the setup cell, e.g.
   `/kaggle/input/datasets/<user>/vqa-repo`.

   Or attach two inputs — `--part code` (≈6 MB) and `--part images` (≈356 MB).
   The images never change, so later code fixes then cost a 6 MB re-upload
   instead of 362 MB. The setup cell finds both either way.
2. **Attach the model.** *Add Input → Models* → MedGemma 4B instruction-tuned
   (Transformers); MedGemma **1.5** 4B-it works too — both are Gemma 3 and load
   through the same `AutoModelForImageTextToText` path. Paste its path into
   `MODEL_PATH` a few cells down.
3. **Accelerator** → any GPU. *Settings → Accelerator*. The config cell sizes
   the run to whatever card you get: on 40 GB or more it uses the paper's own
   batch 4 × accum 4 with full-precision weights; on 16 GB it falls back to
   batch 1 × accum 16 with 4-bit weights, which needs `bitsandbytes`.

Both paths can be left empty — the cells search the attached inputs — but
pasting them is faster and gives a clearer error when something is not
attached.

> **Import order matters.** Import this notebook *first*, then attach the
> inputs and set the accelerator. Kaggle applies a notebook's stored settings
> on import and will say *"your Accelerator, Data Sources and Internet
> settings have changed"* — this notebook deliberately stores none, so nothing
> of yours is overwritten, but an import after attaching can still reset the
> panel. If your inputs disappear, just add them again.

**Internet on or off is detected automatically.** With it off, the notebook
skips `pip install` and every hub lookup, so the model *must* be attached
(step 2) and Kaggle's preinstalled `transformers` must be ≥ 4.50. With it on
and no model attached, the weights are downloaded instead, which needs an
accepted licence for
[`google/medgemma-4b-it`](https://huggingface.co/google/medgemma-4b-it) and an
`HF_TOKEN` under *Add-ons → Secrets*. The preflight cell checks whichever path
you are on before anything expensive starts.

**What this produces.** Table 3 (dataset statistics, audited against the
paper), Tables 4-6 for the zero-shot baseline and the fine-tuned model on both
the validation split (what the paper reports) and the held-out test split, and
Figure 2.

**On run time.** The paper trains 3 epochs over ~21k samples in ~36 h on an
A6000 (48 GB). The released data gives 14,960 training pairs, so at effective
batch 16 one epoch is 935 optimizer steps and the paper's 3 epochs are **2,805
steps**. A Kaggle session caps at 12 h and a T4 fits roughly 600 of those
steps, so the full run is about **five sessions**, carried across by
checkpoints — the fine-tuning section says how.

`MAX_STEPS = -1` (the default) means those 3 epochs. Setting a number stops
early instead, which is what a smoke test wants and what a single session can
finish; 600 steps is about two thirds of one epoch. That matters more than it
sounds: localisation is the part that needs the epochs. A 600-step adapter
answers the text question correctly and then emits a corner box — it has
learned the `<loc_...>` syntax but not where to put it.

## 0. Setup

In [ ]:
import os, sys, glob, json, shutil, subprocess

# ---------------------------------------------------------------------------
# Optional explicit paths. Leave empty to search the attached inputs.
#   BUNDLE  the code (and the built JSONL, unless it is in a separate input)
#   IMAGES  the chest X-rays, when they were attached as their own dataset
# e.g. "/kaggle/input/datasets/<user>/vqa-repo"
BUNDLE = ""
IMAGES = ""
# ---------------------------------------------------------------------------

def find_inputs(marker, root="/kaggle/input", max_depth=6):
    """Every attached directory containing `marker`, shallowest first.

    Kaggle nests inputs differently depending on how they were attached
    (/kaggle/input/<slug> for some, /kaggle/input/datasets/<user>/<slug> or
    /kaggle/input/models/<user>/<model>/<fw>/<var>/<ver>/ for others), so this
    walks rather than globbing one level.
    """
    hits, base = [], root.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath.count("/") - base >= max_depth:
            dirnames[:] = []
            continue
        # Never descend into the image tree; there are thousands of files.
        dirnames[:] = [d for d in dirnames if d != "train"]
        if os.path.exists(os.path.join(dirpath, marker)):
            hits.append(dirpath)
            dirnames[:] = []
    return sorted(hits, key=len)

def tree(root="/kaggle/input", max_depth=4, limit=40):
    """What is actually attached, files included.

    Two levels of directory names say nothing about why a search failed --
    an un-extracted zip and a dataset that was never attached look identical.
    Showing the files tells the two apart at a glance.
    """
    lines, seen, base = [], 0, root.rstrip("/").count("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count("/") - base
        if depth >= max_depth:
            dirnames[:] = []
        # Never list the image tree; there are thousands of files.
        dirnames[:] = [d for d in dirnames if d != "train"]
        lines.append("  " * depth + (os.path.basename(dirpath) or root) + "/")
        for name in sorted(filenames)[:5]:
            lines.append("  " * (depth + 1) + name)
        if len(filenames) > 5:
            lines.append("  " * (depth + 1) + f"… {len(filenames) - 5} file khác")
        seen += 1
        if seen >= limit:
            lines.append("  …")
            break
    return "\n".join(lines)

def unpack(marker, dest="/kaggle/working/unpacked"):
    """Extract an attached .zip that carries `marker`, if one is still zipped.

    Uploading a bundle through the Kaggle UI usually extracts it, but not
    always -- and a dataset left as a single .zip is indistinguishable from
    one that was never attached. Rather than stop on a zip holding exactly
    what is being looked for, unpack it into the writable tree and go on.
    """
    import zipfile
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        dirnames[:] = [d for d in dirnames if d != "train"]
        for name in filenames:
            if not name.endswith(".zip"):
                continue
            path = os.path.join(dirpath, name)
            try:
                with zipfile.ZipFile(path) as z:
                    inside = z.namelist()
                    if not any(n.startswith(marker) for n in inside):
                        continue
                    out = os.path.join(dest, os.path.splitext(name)[0])
                    # Code and JSONL ride in the same bundle, so this is
                    # reached twice; unpack it once.
                    if os.path.exists(os.path.join(out, marker)):
                        return out
                    print(f"{path} chưa được giải nén; đang bung "
                          f"{len(inside)} file -> {out}")
                    z.extractall(out)
                    return out
            except zipfile.BadZipFile:
                continue
    return None

def locate(marker, what):
    """Every input holding `marker`, unpacking a stray zip if that is all there is."""
    hits = find_inputs(marker)
    if hits:
        return hits
    out = unpack(marker)
    if out:
        return find_inputs(marker, root=out)
    raise SystemExit(
        f"Không input nào chứa {marker} ({what}).\n\n"
        f"Đang attach:\n{tree()}\n\n"
        "Kiểm tra: dataset đã attach chưa, đã trỏ sang version mới nhất chưa, "
        "và Kaggle đã giải nén file .zip chưa.")

# Checking that the .py files merely exist is not enough: an attached dataset
# can carry an older copy of a module that is present but missing a fix, and
# the failure then surfaces half an hour later as a traceback from deep inside
# a library. So look for a snippet each fix introduced.
#
# Snippets, not just function names: some fixes add no new function -- the one
# that moves the box loss onto the GPU is a single line inside an existing
# method -- and a names-only check waves those through.
NEEDS = {
    "modeling":  ["def load_base_model", "def disable_broken_peft_dispatchers",
                  "def adapt_training_kwargs", "def language_side_targets"],
    "evaluate":  ["def diagnose_predictions", "def load_predictions",
                  "def eval_boxes"],
    "train":     ["def latest_checkpoint", "def warn_if_nothing_will_be_saved",
                  "def build_training_arguments"],
    "losses":    ["self.to(logits.device)"],
    "results":   ["def render"],
    "preflight": ["def check_model"],
    "infer":     ["def run"],
    "boxes":     ["def to_canvas", "def match_any", "def greedy_match"],
}

def missing_markers(root):
    """Which of NEEDS a copy of the package under `root` does not have."""
    gone = []
    for mod, markers in NEEDS.items():
        path = os.path.join(root, "vindr_cxr_vqa", mod + ".py")
        if not os.path.exists(path):
            gone.append(f"{mod}.py is missing entirely")
            continue
        src = open(path, encoding="utf-8").read()
        gone += [f"{mod}.py: {m}" for m in markers if m not in src]
    return gone

# The code, the built JSONL and the images may live in one input or in two.
#
# Two attached inputs can both carry vindr_cxr_vqa/ -- an older full bundle
# next to a freshly uploaded code-only dataset -- and picking by path length
# alone is a coin flip between them. So prefer whichever copy actually has the
# functions this notebook needs. When none does, keep the shallowest, so the
# report below names what is missing rather than running yesterday's code.
if BUNDLE:
    CODE_ROOT = BUNDLE
else:
    roots = locate("vindr_cxr_vqa", "code")
    CODE_ROOT = next((r for r in roots if not missing_markers(r)), roots[0])
    if len(roots) > 1:
        print(f"{len(roots)} attached inputs carry vindr_cxr_vqa/; "
              f"chose {CODE_ROOT}")

PROCESSED_ROOT = locate("data/processed/train.jsonl", "JSONL đã build")[0]
IMAGES_ROOT = IMAGES or locate("dataset/train_data/train", "ảnh X-quang")[0]

print("code     :", CODE_ROOT)
print("jsonl    :", PROCESSED_ROOT)
print("images   :", IMAGES_ROOT)

# /kaggle/input is read-only, so work from a writable copy of the code.
WORK = "/kaggle/working/vindr"
os.makedirs(WORK, exist_ok=True)
for item in ("vindr_cxr_vqa", "configs", "requirements.txt", "docs"):
    src = os.path.join(CODE_ROOT, item)
    if not os.path.exists(src):
        continue
    dst = os.path.join(WORK, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

os.chdir(WORK)
sys.path.insert(0, WORK)
print("\ncwd:", os.getcwd())
print(sorted(os.listdir(WORK)))

# Fail now, clearly, if the attached code predates what this notebook needs.
# This re-checks the writable copy rather than CODE_ROOT, so a half-finished
# copy is caught too -- and it is the only report when BUNDLE was set by hand
# or when no attached input is current.
stale = missing_markers(WORK)
if stale:
    raise SystemExit(
        "The attached code is OUT OF DATE. Missing:\n  "
        + "\n  ".join(stale)
        + f"\n\nIt was read from: {CODE_ROOT}\n"
        "Rebuild with `python scripts/make_kaggle_bundle.py --part code`, "
        "upload it as a new version of that dataset, and re-run.")
print("code check: up to date")

In [ ]:
# None = detect automatically; set True/False to force it.
# Detection matters: with Internet off but OFFLINE false, pip spends minutes
# retrying DNS before giving up.
OFFLINE = None

def has_internet(host="pypi.org", port=443, timeout=4):
    import socket
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

if OFFLINE is None:
    OFFLINE = not has_internet()
    print(f"detected: {'no internet' if OFFLINE else 'internet available'}")

if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("offline mode: no pip install, no downloads, no hub lookups")
else:
    print("online mode: packages and weights may be fetched")

In [ ]:
# MedGemma is a Gemma 3 model: it needs a recent transformers.
# With Internet off this is skipped — the preflight cell below checks whether
# Kaggle's preinstalled versions are new enough.
# --retries/--timeout keep a misdetected network from stalling for minutes.
if not OFFLINE:
    %pip install -q -U --retries 1 --timeout 15 \
        "transformers>=4.50" "accelerate>=1.4" "peft>=0.14" \
        "bitsandbytes>=0.45" pyyaml

import torch, transformers
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"vram         {total/2**30:.1f} GB total, {free/2**30:.1f} GB free")
    print("bf16         ", torch.cuda.is_bf16_supported())

In [ ]:
# HF_TOKEN is only needed to DOWNLOAD the gated weights. With the model
# attached from Kaggle, or offline, it is not used at all.
if not OFFLINE:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets")
    except Exception as exc:
        print("No HF_TOKEN secret:", exc)
        print("Fine if you attach the model instead of downloading it.")

os.environ.setdefault("HF_HOME", "/kaggle/working/hf")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

### Where the model weights come from

**Attach MedGemma via *Add Input → Models*, then paste its path into
`MODEL_PATH` below.** Kaggle shows the path on the model's panel in the right
sidebar once it is attached — something like
`/kaggle/input/medgemma/transformers/4b-it/1`.

Leaving `MODEL_PATH` empty is fine too: the cell then searches every attached
input for a directory holding `config.json` plus `*.safetensors`, and falls
back to downloading from HuggingFace when the notebook is online.

With Internet **off** the weights must already be attached — a Kaggle notebook
cannot reach the hub. If no MedGemma model is available to attach, run
`python scripts/fetch_model.py` on a machine with internet (≈8 GB, gated) and
upload the result as a Dataset.

In [ ]:
# ---------------------------------------------------------------------------
# Paste the attached model path here. Add Input -> Models shows it in the
# right-hand panel, e.g.
#   "/kaggle/input/models/<user>/googlemedgemma-1-5-4b-it/transformers/default/1/medgemma-1.5-4b-it"
# Leave empty to search the attached inputs automatically.
MODEL_PATH = ""
# ---------------------------------------------------------------------------

def find_model_dir(root="/kaggle/input"):
    """Any attached directory holding config.json plus safetensors weights."""
    hits = []
    for dirpath, dirnames, files in os.walk(root):
        # The data bundle holds thousands of PNGs; nothing to find in there.
        dirnames[:] = [d for d in dirnames if d != "train"]
        if "config.json" in files and any(f.endswith(".safetensors")
                                          for f in files):
            hits.append(dirpath)
            dirnames[:] = []
    # Prefer a path naming medgemma, then the one with the most weight shards.
    hits.sort(key=lambda p: ("medgemma" not in p.lower(),
                             -len([f for f in os.listdir(p)
                                   if f.endswith(".safetensors")])))
    return hits[0] if hits else None

if MODEL_PATH:
    if not os.path.isdir(MODEL_PATH):
        raise SystemExit(f"MODEL_PATH does not exist: {MODEL_PATH}\n"
                         f"attached inputs: {os.listdir('/kaggle/input')}")
    print("using MODEL_PATH:", MODEL_PATH)
else:
    MODEL_PATH = find_model_dir()
    if MODEL_PATH:
        print("auto-detected model:", MODEL_PATH)
    elif OFFLINE:
        raise SystemExit(
            "Offline mode needs the weights attached, but nothing under "
            "/kaggle/input holds config.json + *.safetensors.\n"
            "Attach MedGemma via Add Input -> Models and set MODEL_PATH.")
    else:
        MODEL_PATH = "google/medgemma-4b-it"
        print("no attached weights; will download", MODEL_PATH)

if os.path.isdir(MODEL_PATH):
    print(sorted(f for f in os.listdir(MODEL_PATH)
                 if not f.startswith("."))[:12])

In [ ]:
# Point the config at the attached bundle and size the run for a 16 GB GPU.
import torch, yaml

CFG_PATH = os.path.join(WORK, "configs", "kaggle.yaml")
with open(os.path.join(WORK, "configs", "default.yaml")) as fh:
    cfg = yaml.safe_load(fh)

cfg["paths"]["dataset_dir"]   = os.path.join(IMAGES_ROOT, "dataset")
cfg["paths"]["processed_dir"] = os.path.join(PROCESSED_ROOT, "data", "processed")
cfg["paths"]["vqa_json"]      = "/kaggle/working/data_v1.json"
cfg["paths"]["output_dir"]    = "/kaggle/working/outputs"

# Point the model at the attached weights (or the hub id, when online).
cfg["model"]["name"] = MODEL_PATH

# Size the run to the GPU Kaggle actually handed us. The paper used batch 4 x
# grad-accum 4 (effective 16) on a 48 GB A6000; keep that effective batch and
# trade batch size for accumulation on smaller cards. Quantizing is only worth
# it when the weights would not otherwise fit.
VRAM = (torch.cuda.get_device_properties(0).total_memory / 2**30
        if torch.cuda.is_available() else 0)
if VRAM >= 40:        # A6000-class or better: the paper's own settings
    batch, accum, four_bit, ckpt = 4, 4, False, False
elif VRAM >= 24:      # L4, A10, 3090
    batch, accum, four_bit, ckpt = 2, 8, False, True
else:                 # T4, P100
    batch, accum, four_bit, ckpt = 1, 16, True, True

cfg["train"]["batch_size"] = batch
cfg["train"]["grad_accum"] = accum
cfg["train"]["load_in_4bit"] = four_bit
cfg["train"]["gradient_checkpointing"] = ckpt
# bf16 on Ampere and newer; fp16 on T4/P100, as in the paper.
cfg["train"]["precision"] = ("bf16" if torch.cuda.is_available()
                             and torch.cuda.is_bf16_supported() else "fp16")
# Generation is far lighter than training, so it can use a wider batch.
INFER_BATCH = 16 if VRAM >= 40 else (8 if VRAM >= 24 else 2)
print(f"sized for {VRAM:.0f} GB: batch {batch} x accum {accum} "
      f"(effective {batch * accum}), 4-bit={four_bit}, "
      f"inference batch {INFER_BATCH}")

# One epoch is ~935 optimizer steps. The paper saves per epoch, but a Kaggle
# session can be cut short at any point, so save by step instead — that is what
# makes `--resume auto` able to pick the run back up in a later session.
cfg["train"]["save_strategy"] = "steps"
cfg["train"]["save_steps"] = 100
cfg["train"]["save_total_limit"] = 3

with open(CFG_PATH, "w") as fh:
    yaml.safe_dump(cfg, fh, sort_keys=False)

print(json.dumps({"paths": cfg["paths"], "model": cfg["model"],
                  "train": cfg["train"]}, indent=2))
assert os.path.isdir(cfg["paths"]["dataset_dir"]), cfg["paths"]["dataset_dir"]
assert os.path.isdir(cfg["paths"]["processed_dir"]), cfg["paths"]["processed_dir"]

### Preflight

Checks package versions, the GPU, the attached weights and the dataset files
in one pass, so a missing piece surfaces now rather than forty minutes into
training. It exits non-zero if anything is blocking.

In [ ]:
# Stop here on a blocking problem rather than failing hours later.
r = subprocess.run([sys.executable, "-m", "vindr_cxr_vqa.preflight",
                    "--config", CFG_PATH], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise SystemExit("Preflight found blocking problems — fix them above "
                     "before running the rest of the notebook.")

## 1. Verify the dataset

Recomputes every statistic in Table 3 from the attached JSONL and audits it
against the published numbers. Four checks are expected to fail — they are
real discrepancies in the released data, documented in `docs/DATA_NOTES.md`.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "-m", "vindr_cxr_vqa.stats",
                      "--config", CFG_PATH],
                     capture_output=True, text=True).stdout)

# Sanity-check the scorer itself: replaying the ground truth as predictions
# must give VQA precision 1.0 and 100% localization at IoU >= 0.5.
print(subprocess.run([sys.executable, "-m", "vindr_cxr_vqa.evaluate",
                      "--config", CFG_PATH, "--oracle", "--split", "val"],
                     capture_output=True, text=True).stdout)

## 2. Zero-shot baseline

The "Pretrained" row of Tables 4-6.

Inference runs over the **validation** split (659 images / 2,637 QA pairs) —
the set the paper reports on. The test split is a subset of it, so the same
predictions score both: validation for the paper-comparable numbers, test for
the held-out result. One pass instead of two.

Expect roughly 1.5-2 h for the full split on a 4-bit T4. Set `N_EVAL` to a few
hundred for a quick pass first; the fine-tuned run below reuses the same cap,
so the two rows stay comparable.

In [ ]:
N_EVAL = None      # cap the QA pairs for a quick pass; None = whole split

OUT = "/kaggle/working/outputs"
PRED_BASE = f"{OUT}/preds_val_base.jsonl"
PRED_FT   = f"{OUT}/preds_val_ft.jsonl"

cmd = [sys.executable, "-m", "vindr_cxr_vqa.infer", "--config", CFG_PATH,
       "--split", "val", "--batch-size", str(INFER_BATCH),
       "--out", PRED_BASE]
if N_EVAL:
    cmd += ["--limit", str(N_EVAL)]
!{" ".join(cmd)}

In [ ]:
def score(pred, tag):
    """Score one prediction file against both splits."""
    for split in ("val", "test"):
        r = subprocess.run(
            [sys.executable, "-m", "vindr_cxr_vqa.evaluate",
             "--config", CFG_PATH, "--pred", pred, "--split", split,
             "--out", f"{OUT}/eval_{split}_{tag}.json"],
            capture_output=True, text=True)
        print(r.stdout or r.stderr)

score(PRED_BASE, "base")

## 3. Fine-tune

LoRA r=16, alpha=32, dropout=0.1, AdamW at 2e-5, weight decay 0.01, effective
batch 16, with the multi-task objective `L = L_VQA + 0.5 * L_bbox` where the
box term is a differentiable soft-GIoU read off the coordinate digit tokens.

**Checkpoints.** A checkpoint is written every 100 optimizer steps into
`/kaggle/working/outputs/medgemma-lora/checkpoint-N`, and `--resume auto` picks
up the newest one. So if a session is cut short, re-run this cell in a new
session and training continues rather than restarting — but **only if you saved
the output**: `/kaggle/working` is wiped when a session ends. Use *Save Version
→ Save & Run All*, or download `medgemma-lora.zip` from the last cell and
re-attach it as a Dataset.

**Spreading 2,805 steps over several sessions.** Each session: *Save Version →
Save & Run All*, then in the next session *Add Input* → this notebook's own
previous output. The cell below copies any `medgemma-lora` it finds under
`/kaggle/input` into place, and `--resume auto` continues from the newest
checkpoint. The cosine schedule is rebuilt for the full 2,805-step horizon and
fast-forwarded, so a run that previously decayed to ~0 at its own cap picks the
learning rate back up instead of continuing at zero.

Set `RESUME = False` to start over — worth it only when a change makes the
saved optimizer state wrong, such as a different learning rate or a different
set of LoRA targets.

In [ ]:
# -1 runs the paper's 3 epochs: 14,960 samples at effective batch 16 is 935
# steps an epoch, so 2,805 in total. That is several sessions' worth -- see
# the note above -- and the point of resuming is to spread it across them.
# Set a number instead to stop early (600 is roughly two thirds of an epoch).
MAX_STEPS = -1

# False starts training from scratch even when a previous run is attached.
# Worth it only if you changed something a resumed optimizer state would
# contradict, such as the learning rate or which modules LoRA wraps.
RESUME = True

ADAPTER = f"{OUT}/medgemma-lora"

if RESUME:
    # /kaggle/working is wiped between sessions, so a previous run comes back
    # as an attached input and has to be copied into place before
    # --resume auto can see it.
    for src in glob.glob("/kaggle/input/*/medgemma-lora"):
        print("found previous checkpoints:", src)
        shutil.copytree(src, ADAPTER, dirs_exist_ok=True)

done = sorted(glob.glob(f"{ADAPTER}/checkpoint-*"),
              key=lambda s: int(s.rsplit("-", 1)[1]))
if done:
    at = int(done[-1].rsplit("-", 1)[1])
    print(f"resuming at step {at}" + (f" of {MAX_STEPS}" if MAX_STEPS > 0
                                      else " of 2,805 (3 epochs)"))
    # The cosine schedule is rebuilt for the new horizon and fast-forwarded,
    # so a run that previously decayed to ~0 at its own cap picks the
    # learning rate back up rather than continuing at zero.

!{sys.executable} -m vindr_cxr_vqa.train --config {CFG_PATH} \
    --output-dir {ADAPTER} --max-steps {MAX_STEPS} \
    --resume {"auto" if RESUME else "none"}

In [ ]:
# What actually got saved.
for p in sorted(glob.glob(f"{ADAPTER}/checkpoint-*"),
                key=lambda s: int(s.rsplit("-", 1)[1])):
    size = sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(p) for f in fs) / 2**20
    print(f"{os.path.basename(p):<20} {size:7.1f} MB")

adapter_files = [f for f in os.listdir(ADAPTER) if not f.startswith("check")]
print("\nfinal adapter:", adapter_files)
assert any(f.startswith("adapter_model") for f in adapter_files), \
    "no adapter weights were written — training did not finish"

## 4. Final results

Generates with the fine-tuned adapter over the same split and cap as the
baseline, then scores Table 4 (VQA), Table 5 (box detection at IoU >= 0.3) and
Table 6 (per-image localization quality) on **both** splits:

* **validation** — 659 images, directly comparable to the paper's Tables 4-6;
* **test** — 300 held-out images, the final result.

In [ ]:
cmd = [sys.executable, "-m", "vindr_cxr_vqa.infer", "--config", CFG_PATH,
       "--split", "val", "--adapter", ADAPTER,
       "--batch-size", str(INFER_BATCH), "--out", PRED_FT]
if N_EVAL:
    cmd += ["--limit", str(N_EVAL)]
!{" ".join(cmd)}

score(PRED_FT, "ft")

In [ ]:
# Final tables, baseline vs fine-tuned, with the published numbers alongside.
from IPython.display import Markdown, display

for split in ("val", "test"):
    subprocess.run([sys.executable, "-m", "vindr_cxr_vqa.results",
                    "--base", f"{OUT}/eval_{split}_base.json",
                    "--ft",   f"{OUT}/eval_{split}_ft.json",
                    "--split", split,
                    "--out",  f"{OUT}/results_{split}.md"],
                   capture_output=True, text=True)
    display(Markdown(open(f"{OUT}/results_{split}.md").read()))

## 5. Qualitative results

Figure 2 of the paper: ground truth in green, zero-shot prediction in yellow,
fine-tuned prediction in red.

In [ ]:
!{sys.executable} -m vindr_cxr_vqa.visualize --config {CFG_PATH} \
    --pred-base {PRED_BASE} --pred-ft {PRED_FT} \
    --split val --n 6 --out {OUT}/figures

from IPython.display import Image, display
for p in sorted(glob.glob(f"{OUT}/figures/*.png"))[:6]:
    print(p)
    display(Image(filename=p))

## 6. Save outputs

In [ ]:
# /kaggle/working is wiped when the session ends — archive what matters so it
# survives as notebook output, and so a later session can resume from it.
shutil.make_archive("/kaggle/working/medgemma-lora", "zip", ADAPTER)
print("adapter archive: "
      f"{os.path.getsize('/kaggle/working/medgemma-lora.zip') / 2**20:.1f} MB")

print("\nmetrics and predictions:")
for f in sorted(glob.glob(f"{OUT}/*.json") + glob.glob(f"{OUT}/*.md")
                + glob.glob(f"{OUT}/*.jsonl")):
    print(f"  {os.path.basename(f):<28} "
          f"{os.path.getsize(f) / 2**10:8.1f} KB")

print("\nTo continue training in another session: Save Version -> Save & Run "
      "All, then attach this notebook's output as an input next time; the "
      "train cell copies medgemma-lora back and resumes.")